In [ ]:
"""
Logistic regression analysis: stoichiometry features → drug target status.

Demonstrates that stoichiometry features carry statistically significant
signal for predicting drug target status, independent of the full ML
pipeline, using standard logistic regression with formal hypothesis tests.

Uses statsmodels for reproducible, publication-ready inference.

Usage
-----
    Paths are configured in the Configuration section below.
    Just hit Run in VS Code — no command line arguments needed.

    Paths can also be overridden from the command line:
        python stoich_logistic_regression.py \
            --features-file <path> --labels-file <path> --outdir <path>

Notes
-----
    - The features file (hypergraph_features.csv) is generated by
      cp_hypergraph_features.ipynb and contains 34 hypergraph-derived
      features for 3,394 proteins. It does NOT contain labels.
    - Labels are merged from a separate lookup table produced by
      the label-preparation pipeline (ChEMBL for drug targets,
      OGEE/DepMap for essentiality).
    - Features are standardised (zero mean, unit variance) before
      fitting so that odds ratios are per-SD and comparable across
      features.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import chi2

# ---------------------------------------------------------------------------
# Configuration — edit these paths to match your local setup
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path("/Users/anitaapplegarth/github/dphil/protein_complexes")

FEATURES_FILE = PROJECT_ROOT / "data/lookup_tables/cp/hypergraph_features.csv"
LABELS_FILE   = PROJECT_ROOT / "data/lookup_tables/cp_drug_target_hpa.csv"
OUTPUT_DIR    = PROJECT_ROOT / "python/analysis/stoichiometry/regression"

PROTEIN_ID_COL = "ProteinId"
LABEL_COL = "target"

# Stoichiometry features — the 6 features that use stoichiometric information
STOICH_FEATURES = [
    "stoich_WeightedTriangles",      # triangle count weighted by stoichiometry
    "stoich_AvgNeighbourDegreeStoich",  # average neighbour degree (stoich-weighted)
    "stoich_RangeComplexSize",       # range of complex size (by stoichiometry)
    "stoich_MedComplexSize",         # median complex size (by stoichiometry)
    "stoich_MedianRatio",            # median stoichiometric ratio across complexes
    "stoich_RangeRatio",             # range of stoichiometric ratio across complexes
]

mpl.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "figure.dpi": 150,
})


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def summarise_logit(model_result, feature_names):
    """Extract a tidy summary table from a statsmodels Logit result."""
    params = model_result.params
    conf = model_result.conf_int()
    rows = []
    for i, name in enumerate(feature_names):
        rows.append({
            "Feature": name,
            "Coefficient": params.iloc[i],
            "Std Error": model_result.bse.iloc[i],
            "z": model_result.tvalues.iloc[i],
            "p-value": model_result.pvalues.iloc[i],
            "Odds Ratio": np.exp(params.iloc[i]),
            "OR 95% CI Lower": np.exp(conf.iloc[i, 0]),
            "OR 95% CI Upper": np.exp(conf.iloc[i, 1]),
        })
    return pd.DataFrame(rows)


def plot_forest(df, title, outpath):
    """Forest plot of odds ratios with 95% CIs."""
    df = df[df["Feature"] != "const"].iloc[::-1].copy()

    fig, ax = plt.subplots(figsize=(8, max(3, len(df) * 0.8 + 1.5)))
    y_pos = np.arange(len(df))

    ax.errorbar(
        df["Odds Ratio"], y_pos,
        xerr=[df["Odds Ratio"] - df["OR 95% CI Lower"],
              df["OR 95% CI Upper"] - df["Odds Ratio"]],
        fmt="o", color="#2c5f8a", ecolor="#2c5f8a",
        capsize=5, markersize=8, linewidth=2,
    )
    ax.axvline(x=1, color="grey", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df["Feature"])
    ax.set_xlabel("Odds Ratio (95% CI)")
    ax.set_title(title)

    for i, (_, row) in enumerate(df.iterrows()):
        p = row["p-value"]
        stars = ("***" if p < 0.001 else
                 "**" if p < 0.01 else
                 "*" if p < 0.05 else "ns")
        ax.annotate(
            f'OR={row["Odds Ratio"]:.2f} ({stars})',
            xy=(row["OR 95% CI Upper"], i),
            xytext=(10, 0), textcoords="offset points",
            fontsize=12, va="center",
        )

    plt.tight_layout()
    plt.savefig(outpath, bbox_inches="tight")
    plt.close()
    print(f"  Forest plot saved: {outpath}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    # Use the hardcoded defaults from the Configuration section above.
    # These can be overridden via command line when running as a script,
    # but are used directly when running in Jupyter / VS Code interactive.
    features_file = FEATURES_FILE
    labels_file = LABELS_FILE
    protein_id = PROTEIN_ID_COL
    label = LABEL_COL
    features = STOICH_FEATURES
    outdir = OUTPUT_DIR

    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------
    # Load and merge data
    # ------------------------------------------------------------------
    feat_df = pd.read_csv(features_file)
    labels_df = pd.read_csv(labels_file)
    print(f"Features: {len(feat_df)} proteins from {features_file}")
    print(f"Labels:   {len(labels_df)} proteins from {labels_file}")

    # Validate columns
    if protein_id not in feat_df.columns:
        print(f"ERROR: '{protein_id}' not in features file. "
              f"Available: {sorted(feat_df.columns.tolist())}")
        sys.exit(1)
    if protein_id not in labels_df.columns:
        print(f"ERROR: '{protein_id}' not in labels file. "
              f"Available: {sorted(labels_df.columns.tolist())}")
        sys.exit(1)
    if label not in labels_df.columns:
        print(f"ERROR: '{label}' not in labels file. "
              f"Available: {sorted(labels_df.columns.tolist())}")
        sys.exit(1)

    df = feat_df.merge(
        labels_df[[protein_id, label]],
        on=protein_id, how="inner"
    )
    print(f"Merged:   {len(df)} proteins with both features and labels")

    # Check feature columns
    missing = [c for c in features if c not in df.columns]
    if missing:
        print(f"ERROR: feature columns not found: {missing}")
        stoich_cols = [c for c in df.columns if "stoich" in c.lower()
                       or "unique" in c.lower()]
        print(f"Available stoichiometry-related columns: {stoich_cols}")
        sys.exit(1)

    # Drop rows with NaN in features or label
    n_before = len(df)
    df = df.dropna(subset=features + [label])
    if len(df) < n_before:
        print(f"Dropped {n_before - len(df)} rows with NaN values")

    y = df[label].astype(float)
    print(f"\nLabel distribution: {int(y.sum())} positive / "
          f"{int(len(y) - y.sum())} negative "
          f"({100 * y.mean():.1f}% positive rate)\n")

    # Standardise features (z-score) so ORs are per-SD and comparable
    X_raw = df[features].astype(float)
    X_z = (X_raw - X_raw.mean()) / X_raw.std()
    X_z = sm.add_constant(X_z)

    all_results = []

    # ------------------------------------------------------------------
    # 1. Univariate logistic regressions
    # ------------------------------------------------------------------
    print("=" * 70)
    print("UNIVARIATE LOGISTIC REGRESSIONS (standardised features)")
    print("=" * 70)

    for feat in features:
        Xi = sm.add_constant(X_z[feat])
        model = sm.Logit(y, Xi).fit(disp=False)
        res = summarise_logit(model, ["const", feat])
        res["Model"] = f"Univariate: {feat}"

        print(f"\n--- {feat} ---")
        print(f"  Pseudo R²:      {model.prsquared:.4f}")
        print(f"  Log-likelihood: {model.llf:.2f}")
        feat_row = res[res["Feature"] == feat].iloc[0]
        print(f"  β = {feat_row['Coefficient']:.4f}, "
              f"OR = {feat_row['Odds Ratio']:.3f} "
              f"[{feat_row['OR 95% CI Lower']:.3f}, "
              f"{feat_row['OR 95% CI Upper']:.3f}], "
              f"p = {feat_row['p-value']:.2e}")

        all_results.append(res)

    # ------------------------------------------------------------------
    # 2. Multivariate logistic regression
    # ------------------------------------------------------------------
    print(f"\n{'=' * 70}")
    print("MULTIVARIATE LOGISTIC REGRESSION (standardised features)")
    print("=" * 70)

    model_multi = sm.Logit(y, X_z).fit(disp=False)
    res_multi = summarise_logit(model_multi, ["const"] + features)
    res_multi["Model"] = "Multivariate"
    all_results.append(res_multi)

    print(model_multi.summary2())

    # Variance inflation factors (check for multicollinearity)
    print("\nVariance Inflation Factors:")
    for i, feat in enumerate(features):
        vif = variance_inflation_factor(X_z.values, i + 1)  # +1 skips const
        print(f"  {feat}: VIF = {vif:.2f}")

    # ------------------------------------------------------------------
    # 3. OLS linear probability model (for comparison)
    # ------------------------------------------------------------------
    print(f"\n{'=' * 70}")
    print("OLS LINEAR PROBABILITY MODEL (HC1 robust SEs, for comparison)")
    print("=" * 70)

    ols_model = sm.OLS(y, X_z).fit(cov_type="HC1")
    print(ols_model.summary2())
    print("\nNote: OLS coefficients = change in P(drug_target=1) per 1 SD "
          "increase. HC1 robust standard errors correct for "
          "heteroscedasticity inherent in binary outcomes.")

    # ------------------------------------------------------------------
    # 4. Likelihood ratio test: full model vs intercept-only
    # ------------------------------------------------------------------
    print(f"\n{'=' * 70}")
    print("LIKELIHOOD RATIO TEST (full model vs intercept-only)")
    print("=" * 70)

    null_model = sm.Logit(
        y, sm.add_constant(np.ones(len(y)))
    ).fit(disp=False)
    lr_stat = 2 * (model_multi.llf - null_model.llf)
    lr_df = len(features)
    lr_pval = chi2.sf(lr_stat, lr_df)
    print(f"  LR statistic:       {lr_stat:.2f}")
    print(f"  Degrees of freedom: {lr_df}")
    print(f"  p-value:            {lr_pval:.2e}")
    sig = ("Stoichiometry features jointly significant" if lr_pval < 0.05
           else "Not significant")
    print(f"  → {sig}")

    # ------------------------------------------------------------------
    # 5. Save outputs
    # ------------------------------------------------------------------
    all_df = pd.concat(all_results, ignore_index=True)
    csv_path = outdir / "stoich_logistic_regression_summary.csv"
    all_df.to_csv(csv_path, index=False)
    print(f"\nResults table saved: {csv_path}")

    forest_path = outdir / "stoich_logistic_regression_forest.png"
    plot_forest(
        res_multi,
        "Stoichiometry → Drug Target\n(Multivariate Logistic Regression)",
        forest_path,
    )

    # ------------------------------------------------------------------
    # 6. Interpretation guide
    # ------------------------------------------------------------------
    print(f"\n{'=' * 70}")
    print("INTERPRETATION GUIDE")
    print("=" * 70)
    print("""
Features are standardised (zero mean, unit variance) before fitting,
so coefficients and odds ratios are comparable across features.

Logistic model:  log(p / (1-p)) = β₀ + β₁x₁ + β₂x₂ + ...

  Coefficient (β) : change in log-odds per 1 SD increase in feature
  Odds Ratio (OR) : exp(β) — multiplicative change in odds per 1 SD
      OR > 1  →  higher feature value increases drug target probability
      OR < 1  →  higher feature value decreases drug target probability
      OR = 1  →  no association
  95% CI          : if it excludes 1.0, the effect is significant at α=0.05
  p-value         : Wald test of H₀: β = 0
      * p < 0.05   ** p < 0.01   *** p < 0.001

Likelihood ratio test compares the full model (all stoichiometry features)
against an intercept-only model. A significant result means the features
jointly improve prediction beyond the base rate.

Pseudo R² (McFadden's) measures improvement in log-likelihood over the
null model; values of 0.2–0.4 are considered excellent for logistic
regression.
    """)


if __name__ == "__main__":
    main()

Features: 3394 proteins from /Users/anitaapplegarth/github/dphil/protein_complexes/data/lookup_tables/cp/hypergraph_features.csv
Labels:   3394 proteins from /Users/anitaapplegarth/github/dphil/protein_complexes/data/lookup_tables/cp_drug_target_hpa.csv
Merged:   3394 proteins with both features and labels

Label distribution: 272 positive / 3122 negative (8.0% positive rate)

UNIVARIATE LOGISTIC REGRESSIONS (standardised features)

--- stoich_WeightedTriangles ---
  Pseudo R²:      0.0014
  Log-likelihood: -945.98
  β = -0.1288, OR = 0.879 [0.738, 1.048], p = 1.50e-01

--- stoich_AvgNeighbourDegreeStoich ---
  Pseudo R²:      0.0008
  Log-likelihood: -946.57
  β = -0.0857, OR = 0.918 [0.793, 1.062], p = 2.49e-01

--- stoich_RangeComplexSize ---
  Pseudo R²:      0.0000
  Log-likelihood: -947.29
  β = 0.0124, OR = 1.012 [0.906, 1.132], p = 8.27e-01

--- stoich_MedComplexSize ---
  Pseudo R²:      0.0001
  Log-likelihood: -947.23
  β = -0.0290, OR = 0.971 [0.845, 1.117], p = 6.85e-01

-